# Lesson 34: LangGraph — building a self-correcting RAG system 🕸️
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain why some AI workflows need **graphs** (branches and loops), not just straight chains
2. Build LangGraph apps from **state**, **nodes** and **edges**, including **conditional edges** and **loops**
3. Add **memory** with a checkpointer
4. Build a **RAG system with LangChain + LangGraph** that grades its own retrieval and retries with a better query

**⏱️ Timing:** first graph 15' | loops & memory 10' | self-correcting RAG 30' | review 5'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🚇 An analogy: a train line vs a metro map
A **chain** (Lesson 30) is a single train line: A → B → C, always in order.
Real tasks need a **metro map**: *if* the answer looks wrong, go back; *if* the user needs maths, take the calculator line; *loop* until done.

**LangGraph** (from the LangChain team) builds these maps:
| Part | Meaning | Metro analogy |
|---|---|---|
| **State** | A shared notebook every step can read and update | Your ticket, stamped at each station |
| **Node** | A Python function (often calls an LLM) | A station |
| **Edge** | Which node runs next | A track |
| **Conditional edge** | Choose the next node based on the state | A junction with signals |

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install langgraph langchain langchain-ollama langchain-community langchain-huggingface langchain-text-splitters faiss-cpu
import warnings; warnings.filterwarnings("ignore")
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
llm = ChatOllama(model=MODEL, temperature=0)

## Part 1: Your first graph — a homework checker ✏️
The graph answers a maths question, a **checker** node verifies it with Python, and if it's wrong we **loop back** with feedback (max 3 tries).

In [ ]:
class HomeworkState(TypedDict):
    question: str
    expected: float
    answer: str
    attempts: int
    feedback: str

def solve(state):
    hint = f"Your previous answer was wrong: {state['feedback']}. Try again carefully. " if state["feedback"] else ""
    reply = llm.invoke(f"{hint}{state['question']} Give only the final number.").content
    print(f"  🧮 attempt {state['attempts'] + 1}: {reply.strip()}")
    return {"answer": reply, "attempts": state["attempts"] + 1}

def check(state):
    import re
    nums = re.findall(r"-?\d+\.?\d*", state["answer"].replace(",", ""))
    ok = bool(nums) and abs(float(nums[-1]) - state["expected"]) < 1e-6
    return {"feedback": "" if ok else f"you said {state['answer'].strip()}"}

def route(state):
    if state["feedback"] == "": return "correct"
    return "give up" if state["attempts"] >= 3 else "retry"

g = StateGraph(HomeworkState)
g.add_node("solve", solve)
g.add_node("check", check)
g.add_edge(START, "solve")
g.add_edge("solve", "check")
g.add_conditional_edges("check", route, {"correct": END, "retry": "solve", "give up": END})
homework = g.compile()

result = homework.invoke({"question": "What is 48 × 25 − 17?", "expected": 1183, "answer": "", "attempts": 0, "feedback": ""})
print("✅ correct!" if result["feedback"] == "" else "❌ gave up", "after", result["attempts"], "attempt(s)")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(homework.get_graph().draw_mermaid_png()))     # needs internet (renders with mermaid.ink)
except Exception:
    print(homework.get_graph().draw_mermaid())

### 🧩 Exercise 1
1. Try harder questions (e.g. `"What is 17% of 2,350?"`, expected 399.5). How often does the loop rescue a wrong first answer?
2. Change the router so it gives up after **5** attempts. Why do loops always need a limit?

## Part 2: Memory with a checkpointer 💾
A **checkpointer** saves the state after every step, per conversation **thread**. That gives agents memory — and lets you pause and resume.

In [ ]:
from langgraph.graph import MessagesState
from langgraph.checkpoint.memory import InMemorySaver

def chatbot(state: MessagesState):
    return {"messages": [llm.invoke([("system", "You are a friendly study buddy. Be brief.")] + state["messages"])]}

cg = StateGraph(MessagesState)
cg.add_node("chatbot", chatbot)
cg.add_edge(START, "chatbot"); cg.add_edge("chatbot", END)
buddy = cg.compile(checkpointer=InMemorySaver())

thread = {"configurable": {"thread_id": "rua"}}
for msg in ["Hi, I'm Rua and my favourite subject is chemistry.", "What's my favourite subject?"]:
    out = buddy.invoke({"messages": [("user", msg)]}, thread)
    print(f"🧑 {msg}\n🤖 {out['messages'][-1].content}\n")
print("Another thread:", buddy.invoke({"messages": [("user", "What's my name?")]}, {"configurable": {"thread_id": "someone-else"}})["messages"][-1].content)

## Part 3: A self-correcting RAG system with LangChain + LangGraph 🔁📚
Plain RAG (Lesson 32) retrieves **once** and hopes for the best. This system checks its own work:
```
START → retrieve → grade documents ──relevant──► generate → END
              ▲            │
              │      not relevant
              └── rewrite question ◄┘   (at most 2 rewrites, then answer "I don't know")
```
LangChain gives us the pieces (retriever, LLM, prompts); LangGraph wires up the branches and loop.

In [ ]:
HANDBOOK = {
"attendance.md": """# Attendance
Students must arrive by 8:40am. Students arriving after 8:40am must sign in at the main office and collect a late slip.
If a student is sick, a parent must phone the attendance line (09 555 0101) before 9:00am or send a message through the school app.
Students leaving early must bring a signed note and sign out at the office. Unexplained absences are followed up by the dean.""",
"devices.md": """# Devices and phones
Phones must be switched off and stored in lockers during class time. Teachers may allow devices for a specific learning activity.
Phones may be used at interval and lunch in outdoor areas only. Confiscated phones can be collected from the office at 3:15pm.
Every Year 9 and 10 student needs a laptop or Chromebook with at least 6 hours of battery. The school Wi-Fi network is called StudentNet.""",
"library.md": """# Library
The library is open 8:00am to 4:30pm on weekdays and is closed on weekends and public holidays.
Students may borrow up to 5 books for three weeks. Overdue books incur no fines, but borrowing is paused until books are returned.
Quiet study rooms can be booked at the front desk for groups of up to four students. Printing costs 10 cents per page.""",
"uniform.md": """# Uniform
The school jersey must be worn from May to September. Black leather shoes are required; sports shoes are only for PE.
Hats must be worn outdoors in Terms 1 and 4 (sun-smart policy). Jewellery is limited to a watch and one pair of small studs.
Uniform items can be bought from the uniform shop, open Tuesdays and Thursdays 8:00–9:00am.""",
"assessment.md": """# Assessment
Assessment extensions must be requested from the subject teacher at least two school days before the due date.
Students who are sick on an assessment day must provide a medical certificate within five days.
Using AI tools in an assessment is only allowed when the teacher's task sheet says so, and any AI use must be acknowledged.
Plagiarism, including copying AI-generated work without acknowledgement, results in the assessment not being awarded.""",
"sport.md": """# Sport and activities
Sports trials are held in the first two weeks of each term. The senior basketball team trains Mondays and Wednesdays 3:30–5pm.
Students need a signed permission slip and must pay the activity fee before attending any trip.
The swimming sports day is in week 5 of Term 1. Togs, a towel and sunscreen are required.""",
}
import os
os.makedirs("handbook", exist_ok=True)
for name, text in HANDBOOK.items():
    open(f"handbook/{name}", "w").write(text)
print("Wrote", len(HANDBOOK), "handbook files:", list(HANDBOOK))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

docs = DirectoryLoader("handbook", glob="*.md", loader_cls=TextLoader).load()
splits = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50).split_documents(docs)
retriever = FAISS.from_documents(splits, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")).as_retriever(search_kwargs={"k": 3})
print(len(splits), "chunks indexed")

In [ ]:
class RAGState(TypedDict):
    question: str          # the user's original question
    search_query: str      # what we actually search for (may be rewritten)
    documents: list
    relevant: bool
    rewrites: int
    answer: str

def retrieve(state):
    docs = retriever.invoke(state["search_query"])
    print(f"🔎 retrieve: '{state['search_query']}' → {[d.metadata['source'].split('/')[-1] for d in docs]}")
    return {"documents": docs}

def grade(state):
    text = "\n".join(d.page_content for d in state["documents"])
    verdict = llm.invoke(f"Question: {state['question']}\n\nDocuments:\n{text}\n\n"
                         "Do these documents contain the information needed to answer the question? Reply YES or NO only.").content
    relevant = "YES" in verdict.upper()
    print(f"🧑‍⚖️ grade: {'relevant' if relevant else 'NOT relevant'}")
    return {"relevant": relevant}

def rewrite(state):
    new_q = llm.invoke(f"Rewrite this question as a short search query for a school handbook, using different words. "
                       f"Reply with the query only.\nQuestion: {state['question']}").content.strip()
    print(f"✍️ rewrite: '{new_q}'")
    return {"search_query": new_q, "rewrites": state["rewrites"] + 1}

def generate(state):
    if not state["relevant"]:
        return {"answer": "I couldn't find that in the handbook — please ask the school office."}
    sources = "\n".join(f"({d.metadata['source'].split('/')[-1]}) {d.page_content}" for d in state["documents"])
    answer = llm.invoke(f"Answer using only these sources and name the source file.\n\n{sources}\n\nQuestion: {state['question']}").content
    return {"answer": answer}

def decide(state):
    if state["relevant"]: return "generate"
    return "rewrite" if state["rewrites"] < 2 else "generate"

rg = StateGraph(RAGState)
for name, fn in [("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("generate", generate)]:
    rg.add_node(name, fn)
rg.add_edge(START, "retrieve")
rg.add_edge("retrieve", "grade")
rg.add_conditional_edges("grade", decide, {"generate": "generate", "rewrite": "rewrite"})
rg.add_edge("rewrite", "retrieve")
rg.add_edge("generate", END)
smart_rag = rg.compile()

def ask_handbook(q):
    out = smart_rag.invoke({"question": q, "search_query": q, "documents": [], "relevant": False, "rewrites": 0, "answer": ""})
    print(f"🤖 {out['answer']}\n")

ask_handbook("Is my cellphone allowed during maths?")
ask_handbook("Where can I buy a new school jumper?")
ask_handbook("Does the school have a swimming pool for lunchtime swims?")

In [ ]:
try:
    display(Image(smart_rag.get_graph().draw_mermaid_png()))
except Exception:
    print(smart_rag.get_graph().draw_mermaid())

### 🧩 Exercise 2
1. Read the traces. When did the graph **rewrite** the question? Did the rewrite help?
2. Add a **hallucination check** node after `generate`: ask the LLM "Is every fact in this answer supported by the sources? YES/NO" and loop back to `generate` (once) if NO.
3. Use `smart_rag.stream(..., stream_mode="updates")` to watch each node's output as it happens.

### 🚀 Challenge: route by topic
Add a **router** node at the start: if the question is about school rules → RAG branch; if it's a maths question → a calculator branch; otherwise → a polite "I only help with school questions" node.

## 📌 Summary
- **LangGraph** = state + nodes + edges; **conditional edges** create branches and **loops**
- **Checkpointers** give memory per thread
- A self-correcting RAG **grades** its retrieval and **rewrites** the query before answering — a first step toward **agentic RAG** (Lesson 40)